# 01 — Getting Started with the Sentinel Data Lake

**Goal:** authenticate, connect to the Log Analytics workspace holding `GigamonCcfMcpDemo_CL`, and run the first KQL query.

This notebook is the foundation every other notebook in this repo builds on.

## Setup

Authenticate with Azure CLI (`az login`) and load workspace coordinates from `.env`.

In [ ]:
import os, datetime as dt
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

load_dotenv()
WORKSPACE_ID = os.environ['WORKSPACE_ID']
HOURS = int(os.environ.get('TIMERANGE_HOURS', '24'))
client = LogsQueryClient(DefaultAzureCredential())
TIMESPAN = dt.timedelta(hours=HOURS)

def kql(q: str) -> pd.DataFrame:
    """Run a KQL query and return the first table as a DataFrame."""
    r = client.query_workspace(WORKSPACE_ID, q, timespan=TIMESPAN)
    if r.status != LogsQueryStatus.SUCCESS:
        raise RuntimeError(r.partial_error)
    t = r.tables[0]
    return pd.DataFrame(t.rows, columns=[c for c in t.columns])

## Sanity check — how much Gigamon data do we have?

In [ ]:
df = kql('''
GigamonCcfMcpDemo_CL
| summarize Events=count(),
            FirstSeen=min(TimeGenerated),
            LastSeen=max(TimeGenerated),
            UniqueSources=dcount(src_ip),
            UniqueDestinations=dcount(dst_ip),
            UniqueApps=dcount(app_name)
''')
df

## What protocols / apps are present?

In [ ]:
kql('''
GigamonCcfMcpDemo_CL
| summarize Flows=count() by app_family, protocol
| top 20 by Flows
''')

## Next steps

- `02` — Lateral movement graph
- `03` — JA3 fingerprint hunting
- `04` — Beacon periodicity
- `05` — App-mix dashboard